# On-policy distillation: Teacher model trains a student model

In this tutorial, we take two models, Qwen3-8B and Qwen3-4B, and use
the larger 8B model to teach the smaller 4B model on its generation logprobs.

OPD needs **prompt** log-probs over the student's tokens (prefill with
`max_new_tokens=0`). That is SGLang's native `/generate` API
(`return_logprob=True`, `logprob_start_len=0`). Managed Modal Endpoints only
expose OpenAI-compatible `/v1` and cannot score an existing sequence that way,
so the teacher and student checks use small custom Modal `@app.server`
wrappers around SGLang.

The tutorial follows these steps:

1. **Serve the teacher** (Qwen3-8B) with a custom Modal SGLang Server that exposes `/generate`.
2. **Load a math dataset** (`dapo-math-17k`) and define a verifiable check for `Answer: \boxed{N}`.
3. **Check the base student** (Qwen3-4B) to get a baseline accuracy.
4. **Define a reward function** that POSTs the student tokens to the teacher's `/generate` with `return_logprob=True` and combines the teacher log-probs with a math correctness score.
5. **Train with GRPO + OPD** using `SlimeRecipe` — slime applies a per-token reverse KL penalty from the teacher log-probs on top of the GRPO advantage.
6. **Check the trained student** and compare accuracy before vs after.

### How OPD works

During each rollout step:
1. The student generates a response (math reasoning).
2. The student's token IDs are sent to the teacher's SGLang `/generate` route.
3. The teacher returns per-token log-probabilities.
4. Slime modifies the advantage at each token:

$$A_t = A_t^{\text{GRPO}} - \lambda_{\text{opd}} \cdot (\log \pi_{\text{student}} - \log \pi_{\text{teacher}})$$

The first term pushes toward correct math answers (sparse reward).
The second term pushes toward the teacher's token-level distribution
(dense signal at every position). Together they teach the student
*what* to say (correct answer) and *how* to say it (teacher-like
reasoning).

### Dataset

We use [`zhuzilin/dapo-math-17k`](https://huggingface.co/datasets/zhuzilin/dapo-math-17k),
the same math dataset used by slime's own OPD examples. Each row is a math
problem with a ground-truth integer answer. The model is prompted to
respond with `Answer: \boxed{N}` — the check only verifies whether the
number matches.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import re

import aiohttp
import modal

from modal_training_gym import (
    HuggingFaceDataset,
    Qwen3_4B,
    Qwen3_8B,
    Sample,
    SlimeRecipe,
    TrainConfig,
    endpoint_chat,
    wait_for_server_url,
)

## Serve the teacher on SGLang `/generate`

Managed Modal Endpoints speak OpenAI `/v1` only. OPD's reward path needs
SGLang's native `/generate` so the teacher can return prompt log-probs for
the student's tokens without decoding new ones. The block below is ordinary
Modal user code: `@app.server` + `sglang.launch_server`, then
`TEACHER_GENERATE_URL = f"{url}/generate"`.

In [ ]:
teacher_model = Qwen3_8B()
TEACHER_MODEL_ID = teacher_model.model_name
TEACHER_APP_NAME = "gym-opd-teacher-qwen3-8b"
TEACHER_PORT = 8000
TEACHER_STARTUP_TIMEOUT = 20 * 60

teacher_image = (
    modal.Image.from_registry("lmsysorg/sglang:v0.5.12")
    .entrypoint([])
    .run_commands("rm -rf /root/.cache/huggingface")
    .env(
        {
            "HF_HUB_CACHE": "/root/.cache/huggingface",
            "HF_XET_HIGH_PERFORMANCE": "1",
        }
    )
)
teacher_app = modal.App(TEACHER_APP_NAME)

_teacher_model_id = TEACHER_MODEL_ID
_teacher_port = TEACHER_PORT
_teacher_startup = TEACHER_STARTUP_TIMEOUT

@teacher_app.server(
    image=teacher_image,
    gpu="H100",
    volumes={
        "/root/.cache/huggingface": modal.Volume.from_name(
            "huggingface-cache", create_if_missing=True
        ),
    },
    port=_teacher_port,
    startup_timeout=_teacher_startup,
    scaledown_window=10 * 60,
    exit_grace_period=25,
    target_concurrency=8,
    unauthenticated=True,
    serialized=True,
)
class TeacherServer:
    @modal.enter()
    def start(self):
        import subprocess as _sp
        import time as _time
        import urllib.error as _ue
        import urllib.request as _ur

        cmd = [
            "python",
            "-m",
            "sglang.launch_server",
            "--model-path",
            _teacher_model_id,
            "--served-model-name",
            _teacher_model_id,
            "--host",
            "0.0.0.0",
            "--port",
            str(_teacher_port),
            "--context-length",
            "32768",
            "--mem-fraction-static",
            "0.82",
            "--chunked-prefill-size",
            "8192",
            "--max-running-requests",
            "16",
            "--trust-remote-code",
        ]
        self.proc = _sp.Popen(cmd)
        deadline = _time.monotonic() + _teacher_startup
        health = f"http://127.0.0.1:{_teacher_port}/health"
        while True:
            if self.proc.poll() is not None:
                raise RuntimeError(
                    f"SGLang exited with code {self.proc.returncode} before healthy"
                )
            try:
                with _ur.urlopen(health, timeout=5) as resp:
                    if resp.status == 200:
                        return
            except (_ue.URLError, TimeoutError, OSError):
                pass
            if _time.monotonic() >= deadline:
                raise TimeoutError(f"Teacher SGLang not healthy at {health}")
            _time.sleep(2)

    @modal.exit()
    def stop(self):
        proc = getattr(self, "proc", None)
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=30)

with modal.enable_output():
    teacher_app.deploy()
teacher_url = wait_for_server_url(TeacherServer, label="OPD teacher")
print(f"Teacher URL: {teacher_url}")

# Managed Endpoints stop at /v1. This Server exposes full SGLang, including /generate.
TEACHER_GENERATE_URL = f"{teacher_url}/generate"

Is our teacher good at answering math problems? For a 8B model, it may not be the best
but will surely outperform our smaller, 4B model by a large margin. Every parameter counts!

In [ ]:
response = endpoint_chat(
    teacher_url,
    model=TEACHER_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": (
                "Solve the following math problem step by step. The last "
                "line should be Answer: \\boxed{$Answer}, where $Answer is "
                "the answer.\n\nWhat is 17 * 23?"
            ),
        }
    ],
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
print(response[-200:])

## Dataset

We use a simple math dataset containing competition problems that the LLM is tasked
with answering via a `Answer: \boxed{N}` response. This simple format allows
for deterministic checking!

[Here's the link to `zhuzilin/dapo-math-17k`](https://huggingface.co/datasets/zhuzilin/dapo-math-17k)

Each row contains a `prompt` field with the original chat message and a `label` containing an integer answer.

[Thinking Machines](https://thinkingmachines.ai/blog/on-policy-distillation/) demonstrates that 
using a small number of samples with a larger number of rollouts can be sufficient for OPD.
For this tutorial, we take 100 training samples and hold out 20 for checking.

In [ ]:
class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_column = "prompt"
    output_column = "label"
    output_format = "jsonl"
    apply_chat_template = False

train_dataset = MathDataset(hf_split="train[:100]")
check_dataset = MathDataset(hf_split="train[100:120]")

In [ ]:
rows = check_dataset.load()
for row in rows.select(range(2)):
    prompt = row["prompt"]
    if isinstance(prompt, list):
        prompt = prompt[0]["content"] if prompt else ""
    print(prompt[:200])
    print(f"  label: {row['label']}")
    print()

In [ ]:
def _normalize_answer(answer: str) -> str:
    answer = str(answer).strip()
    answer = answer.split("=")[-1]
    for old, new in [("$", ""), ("\\$", ""), (",", ""), (" ", ""),
                      ("\\text{", ""), ("}", ""), ("\\boxed{", "")]:
        answer = answer.replace(old, new)
    return answer.strip()

def _extract_answer(response: str) -> str:
    match = re.findall(r"(?i)Answer\s*:\s*([^\n]+)", response)
    return match[-1].strip() if match else "[INVALID]"

def _check_math(response: str, label: str) -> bool:
    pred = _normalize_answer(_extract_answer(response))
    gt = _normalize_answer(label)
    try:
        gt = str(int(float(gt)))
    except (ValueError, OverflowError):
        pass
    return pred == gt

def score_math_example(base_url: str, model_id: str, example: dict) -> Sample:
    prompt = example.get("prompt", "")
    if isinstance(prompt, list):
        prompt = prompt[0]["content"] if prompt else ""
    label = example.get("label", "")

    response = endpoint_chat(
        base_url,
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)

    correct = _check_math(response, label)
    pred = _normalize_answer(_extract_answer(response))

    return Sample(
        score=1.0 if correct else 0.0,
        prompt=prompt,
        response=response,
        metadata={"correct": correct, "pred": pred, "label": label},
    )

def run_math_check(base_url: str, model_id: str) -> list[Sample]:
    return [
        score_math_example(base_url, model_id, example)
        for example in check_dataset.load()
    ]

def mean_score(rows: list[Sample]) -> float:
    return sum(row.score for row in rows) / len(rows) if rows else float("nan")

## Baseline check

Qwen3-4B is not in the managed Endpoint catalog, so the student check uses
the same `@app.server` + SGLang pattern as the teacher and ASR tutorials.
Thankfully, our dataset requires
simple-enough answers that a tiny, 4B model should not cause issues for our deterministic parser.
In our own experience, requiring a strict JSON output format can hurt parser reliability.

In [ ]:
base_model = Qwen3_4B()
STUDENT_MODEL_ID = base_model.model_name
STUDENT_APP_NAME = "gym-opd-student-qwen3-4b-check"

def serve_student(
    model_path: str,
    served_model_name: str,
    checkpoints_volume_name: str | None = None,
) -> str:
    student_app = modal.App(STUDENT_APP_NAME)
    volumes = {
        "/root/.cache/huggingface": modal.Volume.from_name(
            "huggingface-cache", create_if_missing=True
        )
    }
    if checkpoints_volume_name:
        volumes["/checkpoints"] = modal.Volume.from_name(
            checkpoints_volume_name, create_if_missing=True
        )

    @student_app.server(
        image=teacher_image,
        gpu="H100",
        volumes=volumes,
        port=TEACHER_PORT,
        startup_timeout=TEACHER_STARTUP_TIMEOUT,
        scaledown_window=10 * 60,
        exit_grace_period=25,
        target_concurrency=4,
        unauthenticated=True,
        serialized=True,
    )
    class StudentServer:
        @modal.enter()
        def start(self):
            import subprocess as _sp
            import time as _time
            import urllib.error as _ue
            import urllib.request as _ur

            self.proc = _sp.Popen(
                [
                    "python",
                    "-m",
                    "sglang.launch_server",
                    "--model-path",
                    model_path,
                    "--served-model-name",
                    served_model_name,
                    "--host",
                    "0.0.0.0",
                    "--port",
                    str(TEACHER_PORT),
                    "--mem-fraction-static",
                    "0.80",
                    "--trust-remote-code",
                ]
            )
            deadline = _time.monotonic() + TEACHER_STARTUP_TIMEOUT
            health = f"http://127.0.0.1:{TEACHER_PORT}/health"
            while True:
                if self.proc.poll() is not None:
                    raise RuntimeError(
                        f"SGLang exited with code {self.proc.returncode} "
                        "before healthy"
                    )
                try:
                    with _ur.urlopen(health, timeout=5) as response:
                        if response.status == 200:
                            return
                except (_ue.URLError, TimeoutError, OSError):
                    pass
                if _time.monotonic() >= deadline:
                    raise TimeoutError(f"Student SGLang not healthy at {health}")
                _time.sleep(2)

        @modal.exit()
        def stop(self):
            proc = getattr(self, "proc", None)
            if proc is not None and proc.poll() is None:
                proc.terminate()
                proc.wait(timeout=30)

    with modal.enable_output():
        student_app.deploy()
    return wait_for_server_url(StudentServer, label="OPD student")

base_url = serve_student(STUDENT_MODEL_ID, STUDENT_MODEL_ID)
print(f"Student URL: {base_url}")

print("--- Checking base student... ---")
base_rows = run_math_check(base_url, STUDENT_MODEL_ID)
base_mean = mean_score(base_rows)
n_correct = sum(1 for row in base_rows if row.metadata.get("correct"))
print(f"Base accuracy: {n_correct}/{len(base_rows)} ({base_mean:.1%})")

In [ ]:
for r in base_rows[:3]:
    status = "CORRECT" if r.metadata["correct"] else "WRONG"
    print(f"[{status}] label={r.metadata['label']}, pred={r.metadata['pred']}")
    print(f"  ...{r.response[-150:]}")
    print()

## Reward function

OPD uses "reverse" KL divergence to grade the student model's output. Remember,
KL divergence D_kl(P || Q) is Sigma_x P(x) * log(P(x) / Q(x)), where P is the behavior distribution
and Q is the target distribution. Forward KL treats the teacher model as P and the student model as Q.
However, the log(P(x) / Q(x)) term would then be weighted by the teacher model's probability distribution P,
making the result being high surprisal on modes unfamiliar to the student model.

Instead, we want to use the reverse KL divergence D_kl(Student || Teacher), where our student model
is treated as the behavior distribution and the teacher model is our target distribution. When the teacher has high surprisal on a 
student mode, the term log(P(x)) - log(Q(x)) will yield a high positive KL divergence to penalize the student model. 
Now, the student model only gets penalized on modes relevant to itself. 

In [ ]:
async def math_opd_rm(args, sample, **kwargs):
    payload = {
        "input_ids": sample.tokens,
        "sampling_params": {
            "temperature": 0,
            "max_new_tokens": 0,
            "skip_special_tokens": False,
        },
        "return_logprob": True,
        "logprob_start_len": 0,
    }
    async with aiohttp.ClientSession() as session:
        async with session.post(
            args.rm_url,
            json=payload,
            allow_redirects=False,
        ) as response:
            response.raise_for_status()
            teacher_response = await response.json()

    label = getattr(sample, "label", "") or ""
    correct = _check_math(sample.response, label)
    sample.math_correct = correct

    return teacher_response

def math_opd_post_process(args, samples, **kwargs):
    """Post-process: store teacher log-probs and return math rewards.

    Delegates to slime's built-in post_process_rewards for the teacher
    log-prob alignment, then overrides the scalar rewards with math
    correctness scores.
    """
    from slime.rollout.on_policy_distillation import post_process_rewards as _opd_post

    _, _ = _opd_post(args, samples, **kwargs)

    math_rewards = []
    for sample in samples:
        correct = getattr(sample, "math_correct", False)
        math_rewards.append(1.0 if correct else -1.0)

    return math_rewards, math_rewards

## Training

The training recipe uses 1 H100 GPU per actor and rollout engine. The actor engine
runs the training and the rollout engine runs the model for inference/forward passes.
You may want to tune the batch size for fitting the memory requirements of your GPU
and increase the samples per prompt parameter for generating more variants per group.
The total_rollouts_per_step is the rollout_batch_size * n_samples_per_prompt, and
the total # of rollouts that occur over a training run is the total_rollouts_per_step * num_rollout.

In [ ]:
training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        custom_rm_function=math_opd_rm,

        gpu_type="H100",
        colocate=True,
        actor_num_gpus_per_node=8,
        rollout_num_gpus=8,
        tensor_model_parallel_size=1,
        sequence_parallel=False,
        rollout_num_gpus_per_engine=1,

        num_rollout=10,
        rollout_batch_size=16,
        n_samples_per_prompt=4,
        rollout_max_response_len=2048,
        rollout_temperature=1.0,

        global_batch_size=16,
        lr=1e-6,
        save_interval=5,
        apply_chat_template_kwargs='{"enable_thinking": true}',

        environment={
            "PYTHONPATH": "/root/Megatron-LM/:/root",
            "CUDA_DEVICE_MAX_CONNECTIONS": "1",
            "NCCL_NVLS_ENABLE": "1",
        },

        extra_config={
            "use_opd": True,
            "opd_type": "sglang",
            "opd_kl_coef": 1.0,
            "custom_reward_post_process_path": (
                "003_on_policy_distillation.math_opd_post_process"
            ),
            "rm_url": TEACHER_GENERATE_URL,
        },
    ),
)

print("--- Starting OPD training... ---")
print(f"  Teacher: {teacher_url}")
print(f"  Student: Qwen3-4B")
print(f"  Dataset: dapo-math-17k (100 problems)")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")
print("--- Training complete ---")

## Check the trained student

Redeploy the custom student server with the checkpoint Volume mounted, then
run the same scoring loop used for the base student.

In [ ]:
trained_model = train_result.hf_model()
print(f"Checkpoint: {trained_model.model_path}")

trained_url = serve_student(
    trained_model.model_path,
    trained_model.model_name,
    train_result.checkpoints_volume,
)
print(f"Trained student URL: {trained_url}")

print("--- Checking trained student... ---")
trained_rows = run_math_check(trained_url, trained_model.model_name)
trained_mean = mean_score(trained_rows)
n_correct = sum(1 for row in trained_rows if row.metadata.get("correct"))
print(f"Trained accuracy: {n_correct}/{len(trained_rows)} ({trained_mean:.1%})")

In [ ]:
for base_r, trained_r in zip(base_rows[:3], trained_rows[:3]):
    label = base_r.metadata["label"]
    b_status = "CORRECT" if base_r.metadata["correct"] else "WRONG"
    t_status = "CORRECT" if trained_r.metadata["correct"] else "WRONG"
    print(f"label={label}")
    print(f"  Base:    [{b_status}] pred={base_r.metadata['pred']}")
    print(f"  Trained: [{t_status}] pred={trained_r.metadata['pred']}")
    print()

## Results

Let's hope you see a positive delta on held-out performance!

In [ ]:
base_correct = sum(1 for row in base_rows if row.metadata.get("correct"))
trained_correct = sum(1 for row in trained_rows if row.metadata.get("correct"))
total = len(base_rows)
print(f"Base student:    {base_correct}/{total} ({base_mean:.1%})")
print(f"Trained student: {trained_correct}/{total} ({trained_mean:.1%})")
print(f"Delta:           {trained_mean - base_mean:+.1%}")

## Next steps

Some cool ways to extend and improve this example:
1. Use a bigger teacher: Qwen3 offers models in the 32B parameter range. 
This model will fit on a 4xH100 GPU setup and can show measurable improvements
on the student model's held-out delta.
2. Tweak the composite reward signal: Try applying a coefficient like *2* to the
binary integer reward signal used in the custom reward function to value correct answers
over student-teacher alignment.
3. Try cross-family distillation: Use a teacher from a different model family (e.g. Kimi K2)
to train our Qwen3-4B student model. You may run into cross-tokenizer differences, so
be careful to only grade logprobs on tokens that exist in both models' vocabularies and
align 1:1 on a per-character basis. 